In [27]:
import sys
import os
current_dir = os.getcwd() # current working directory
project_root = os.path.dirname(current_dir) # 1 directory before it (our project directory)
sys.path.insert(0, project_root) # make it so that this file can use other files in our main project directory (since this file is in notebooks folder)

#### Model Training
(using data imbalance strat 4, scroll below)
extra: complement naive bayes test

In [28]:
import numpy as np
import pandas as pd
import joblib
import data_ingestion
import pipeline
import train_model
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Importing

In [29]:
query = """
    SELECT review_text, sentiment_label
    FROM reviews;
"""
conn, c = data_ingestion.initialize_db()

In [30]:
df = pd.read_sql_query(query, conn)
df

,review_text,sentiment_label
0,baru sekali ini terima brg dr belanja online d...,positive
1,cocok bgt aku sama telur nya. nga Amis menurut...,positive
2,Telornya sudah sampai di rumah dengan kemasan ...,positive
3,Telor sudah diterima dengan baik dan tidak ada...,positive
4,"Alhamdulillah penjual amanah,Telor nya terbaik...",positive
...,...,...
65538,"kwalitas bagus, pokoknya rekomendit deh",positive
65539,Sesuai harga,positive
65540,Sesuai harga,positive
65541,Mantap,positive


Data Cleaning (check train_model for detailed funcs)

In [31]:
train_model.clean_df(df)

Checking for any missing value in columns..
No missing value in column 0
No missing value in column 1
Following columns have missing value present: ['None']
-----------------------
Checking for any duplicate rows..
A total of 7460 duplicate rows have been found in the dataset. Removing them..
0 duplicate rows present in the database.
58083 rows left in the database.
-----------------------


,review_text,sentiment_label
0,baru sekali ini terima brg dr belanja online d...,positive
1,cocok bgt aku sama telur nya. nga Amis menurut...,positive
2,Telornya sudah sampai di rumah dengan kemasan ...,positive
3,Telor sudah diterima dengan baik dan tidak ada...,positive
4,"Alhamdulillah penjual amanah,Telor nya terbaik...",positive
...,...,...
65535,"bola nya sampai dengan aman, baaru dateng lang...",positive
65536,sesuai pesanan keren speeds best seller ☺,positive
65537,"bolanya ok, tampilannya meyakinkan. semoga awe...",positive
65538,"kwalitas bagus, pokoknya rekomendit deh",positive


In [32]:
query_lexicons = """
        SELECT slang, formal
        FROM lexicons;
    """

slang_df = pd.read_sql_query(query_lexicons, conn)
slang_dict = dict(zip(slang_df["slang"], slang_df["formal"]))
df["review_text"] = df["review_text"].apply(lambda x : pipeline.clean_review_text(text=x, data_dict=slang_dict))
df.head(5)

,review_text,sentiment_label
0,baru sekali ini terima bareng dari belanja onl...,positive
1,cocok banget aku sama telur nya enggak amis me...,positive
2,telornya sudah sampai di rumah dengan kemasan ...,positive
3,telor sudah diterima dengan baik dan tidak ada...,positive
4,alhamdulillah penjual amanahtelor nya terbaiks...,positive


In [33]:
df["sentiment_label"] = df["sentiment_label"].map({"negative" : 0, "neutral" : 1, "positive" : 2})

In [34]:
df[["sentiment_label"]].value_counts()

sentiment_label
2                  56506
0                    790
1                    787
Name: count, dtype: int64

Train-Test split

In [35]:
X = df["review_text"]
y = df["sentiment_label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y)

## ---------------------------------

Handling Imbalanced Data

- Should only be done after train-test split to avoid data leakage

Strat 3: Manually setting class_weight penalty during Model initialization.

- Not here, check below

## ---------------------------------

Text Vectorizing

In [36]:
indo_stopwords = stopwords.words("indonesian")
indo_negation_words = {
        "tidak", "bukan", "jangan", "belum", "tanpa", "kurang", "tak", "tiada",
        "tidaklah", "bukanlah", "belumlah", "janganlah", 
        "tidakkah", "bukankah", "belumkah", "bukannya",
        "nggak", "gak", "ga", "ndak", "kagak", "enggak", "ngga"
    }  # apparently nltk has these listed as stopwords lmao. this is to ensure that no negation words are present in the list of stopwords we're using (so that ngram may work properly)
indo_stopwords = [i for i in indo_stopwords if i not in indo_negation_words]

In [37]:
vectorizer = TfidfVectorizer(
        stop_words=indo_stopwords, 
        ngram_range=(1, 2), 
        min_df=3, 
        max_features=20000
    ) # indonesian stop words, unigram and bigram range, minimum document frequency = 3 to ignore one time typo usually, only take into account the top 20000 most impactful words basically

In [38]:
X_train_tfidf = vectorizer.fit_transform(X_train)

c:\Users\User\anaconda3\envs\ecom_nlp_env\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


In [39]:
X_test_tfidf = vectorizer.transform(X_test)

Training

In [40]:
lr_model = LogisticRegression(
        # penalty="l2",
        # class_weight=penalty_weights,
        # C=1.0,
        max_iter=1000,
        random_state=42,
        verbose=3 # just want to see the logging
    ) # l2 (ridge regularization) is the default, class weight balanced is needed since our dependent variable is very unbalanced, regularization strength to 1 since our dataset is very sparse (15000 features is a lot) so C ensures that the model dont overfit (also higher C score reduces the strength), max iter lower than 1000 gives ConvergenceWarning because not enough training iteration for the model to reach optimal convergence when tested
linearsvc_model = LinearSVC(
        # penalty="l2",
        # class_weight=penalty_weights,
        # C=1.0,
        # dual=False,
        multi_class="ovr",
        random_state=42,
        verbose=3
    ) # penalty and class weight follows our LR model. Select the algorithm to either solve the dual or primal optimization problem. Prefer dual=False when n_samples > n_features (from sklearn documentation). our n_samples is around 58k while our n_features is exactly 20k, so dual will be False. ovr is the standard, its also better for text based prediction
cnb_model = ComplementNB(
    # alpha=1.0,
    fit_prior=True
)
# random state for defensive programming practice (reproducibility)
# class_weight and C regularization (and dual for linearsvc) are commented cuz they will be our parameters used in RandomizedSearchCV
# penalty is just deprecated, replaced by l1_ratio
# alpha for complement NB is smoothing parameter. smaller values allow models to be more sensitive to rare words

In [41]:
from scipy.stats import loguniform, uniform

distribution_lr = {
        "C" : loguniform(0.01, 1.0), # inverse regularization parameter, log uniform numbers from 0.01 to 100 (0.01, 0.1, 1, 10, 100)
        "solver" : ["lbfgs"], # optimization/solver algorithm used when training
        "tol" : [0.0001, 0.001], # numerical tolerance, training stops when model performance improvement drops below the number
        "class_weight" : [{0: 10.0, 1: 15.0, 2: 1.0},
                          {0: 12.0, 1: 18.0, 2: 1.0},
                          {0: 14.0, 1: 20.0, 2: 1.0},
                          {0: 16.0, 1: 22.0, 2: 1.0},
                          {0: 18.0, 1: 25.0, 2: 1.0},
                          {0: 20.0, 1: 28.0, 2: 1.0}] # penalty rate for each label if the model got it wrong
}

In [42]:
distribution_svc = [
    # for squared hinge loss
    {
        "C" : loguniform(0.1, 1.0),
        "penalty" : ["l2"],
        "loss" : ["squared_hinge"],
        "dual" : [False],
        "tol" : [0.0001, 0.001],
        "class_weight" : [{0: 12.0, 1: 18.0, 2: 1.0},
                          {0: 15.0, 1: 20.0, 2: 1.0},
                          {0: 18.0, 1: 25.0, 2: 1.0},
                          {0: 22.0, 1: 30.0, 2: 1.0}]
    },
    # for traditional hinge loss
    {
        "C" : loguniform(0.1, 1.0),
        "penalty" : ["l2"],
        "loss" : ["hinge"],
        "dual" : [True], # dual=True is required for traditional hinge loss, though performance may be degraded cuz our n_samples > n_features
        "tol" : [0.0001, 0.001],
        "class_weight": [{0: 12.0, 1: 18.0, 2: 1.0},
                         {0: 15.0, 1: 20.0, 2: 1.0},
                         {0: 18.0, 1: 25.0, 2: 1.0},
                         {0: 22.0, 1: 30.0, 2: 1.0}]
    }
]

In [43]:
distribution_cnb = {
    "alpha" : [0.00001, 0.0001, 0.001, 0.01, 0.1, 1.0],
    
}

In [44]:
lr_rscv = RandomizedSearchCV(lr_model, distribution_lr, n_iter=15, cv=3, scoring="f1_macro", random_state=42, n_jobs=-1, verbose=3)
# n_iter=15 will train 15 random combinations, with cv=3 fold stratified cross-validation, focusing on macro avg f1-score, n_jobs=-1 ensure all CPU cores are used to run, verbose=3 to see most details

In [45]:
linearsvc_rscv = RandomizedSearchCV(linearsvc_model, distribution_svc, n_iter=15, cv=3, scoring="f1_macro", random_state=42, n_jobs=-1, verbose=3)

In [46]:
cnb_rscv = RandomizedSearchCV(cnb_model, distribution_cnb, n_iter=15, cv=3, scoring="f1_macro", random_state=42, n_jobs=-1, verbose=3)

In [47]:
lr_rscv.fit(X_train_tfidf, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LogisticRegre...42, verbose=3)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'C': <scipy.stats....0025F1E904D40>, 'class_weight': [{0: 10.0, 1: 15.0, 2: 1.0}, {0: 12.0, 1: 18.0, 2: 1.0}, ...], 'solver': ['lbfgs'], 'tol': [0.0001, 0.001]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation 

In [48]:
linearsvc_rscv.fit(X_train_tfidf, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
[LibLinear]

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LinearSVC(ran...42, verbose=3)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'C': <scipy.stats....0025F1CEA6FF0>, 'class_weight': [{0: 12.0, 1: 18.0, 2: 1.0}, {0: 15.0, 1: 20.0, 2: 1.0}, ...], 'dual': [False], 'loss': ['squared_hinge'], ...}, {'C': <scipy.stats....0025F1E8992E0>, 'class_weight': [{0: 12.0, 1: 18.0, 2: 1.0}, {0: 15.0, 1: 20.0, 2: 1.0}, ...], 'dual': [True], 'loss': ['hinge'], ...}]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold`

In [49]:
cnb_rscv.fit(X_train_tfidf, y_train)

c:\Users\User\anaconda3\envs\ecom_nlp_env\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 6 is smaller than n_iter=15. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 6 candidates, totalling 18 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",ComplementNB()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'alpha': [1e-05, 0.0001, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intContr

In [50]:
print(f"Best Logistic Regression Parameters found: {lr_rscv.best_params_}")
print(f"Best Linear SVC Parameters found: {linearsvc_rscv.best_params_}")
print(f"Best Complement NB Parameters found: {cnb_rscv.best_params_}")

Best Logistic Regression Parameters found: {'C': np.float64(0.3718364180573207), 'class_weight': {0: 14.0, 1: 20.0, 2: 1.0}, 'solver': 'lbfgs', 'tol': 0.001}
Best Linear SVC Parameters found: {'C': np.float64(0.4789691464869526), 'class_weight': {0: 12.0, 1: 18.0, 2: 1.0}, 'dual': False, 'loss': 'squared_hinge', 'penalty': 'l2', 'tol': 0.0001}
Best Complement NB Parameters found: {'alpha': 0.001}


Predictions

In [51]:
lr_preds = lr_rscv.predict(X_test_tfidf)
linearsvc_preds = linearsvc_rscv.predict(X_test_tfidf)
cnb_preds = cnb_rscv.predict(X_test_tfidf)

Evaluation

In [52]:
print("---Logistic Regression---")
print(classification_report(y_test, lr_preds))
print("---Linear SVC---")
print(classification_report(y_test, linearsvc_preds))
print("---Complement NB---")
print(classification_report(y_test, cnb_preds))

---Logistic Regression---
              precision    recall  f1-score   support

           0       0.36      0.37      0.37       158
           1       0.14      0.32      0.20       157
           2       0.99      0.97      0.98     11302

    accuracy                           0.95     11617
   macro avg       0.50      0.55      0.52     11617
weighted avg       0.97      0.95      0.96     11617

---Linear SVC---
              precision    recall  f1-score   support

           0       0.51      0.28      0.36       158
           1       0.15      0.10      0.12       157
           2       0.98      0.99      0.99     11302

    accuracy                           0.97     11617
   macro avg       0.55      0.46      0.49     11617
weighted avg       0.96      0.97      0.97     11617

---Complement NB---
              precision    recall  f1-score   support

           0       0.17      0.51      0.25       158
           1       0.11      0.22      0.15       157
           2

(I mean, I guess we're still over our Macro F1 Score baseline of 0.326)

- Macro Avg F1 Score Baseline (if a classifier model only predicts positive) = (0.98 + 0 + 0) / 3 = 0.326
- Our Logistic Regression Macro Avg F1 Score: 0.49
- Our Linear SVC Macro Avg F1 Score: 0.46

Precision: Out of all the instances model predicted positive, how many are correct?
Recall (Sensitivity): Out of all actual positives, how many did the model find?
F1-Score is the harmonic mean score of the two.

Precision focuses more on minimizing False Positives.
Recall focuses more on minimizing False Negatives.

Since both precision and recall are equally important here, we can look at our F1-Score. We know our dataset is heavily imbalanced, so judging through our macro avg score, we know that LinearSVC performed better here.

Let's try going further beyond with Logistic Regression

Best Logistic Regression Parameters found: {'C': np.float64(0.3718364180573207), 'class_weight': {0: 14.0, 1: 20.0, 2: 1.0}, 'solver': 'lbfgs', 'tol': 0.001}

In [60]:
distribution_lr2 = {
        "C" : loguniform(0.2, 0.5), # our current best C value is around this range
        "solver" : ["lbfgs"], # keeping this the same
        "tol" : [0.001, 0.0001], # keeping this the same
        "class_weight" : [{0: 12.0, 1: 18.0, 2: 1.0},
                          {0: 13.0, 1: 19.0, 2: 1.0},
                          {0: 14.0, 1: 20.0, 2: 1.0},  # our current best weights
                          {0: 15.0, 1: 21.0, 2: 1.0},
                          {0: 16.0, 1: 22.0, 2: 1.0}]
}

In [61]:
lr_rscv2 = RandomizedSearchCV(lr_model, distribution_lr2, n_iter=20, cv=5, scoring="f1_macro", random_state=42, n_jobs=-1, verbose=3)

In [62]:
lr_rscv2.fit(X_train_tfidf, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LogisticRegre...42, verbose=3)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'C': <scipy.stats....0025F1ED45490>, 'class_weight': [{0: 12.0, 1: 18.0, 2: 1.0}, {0: 13.0, 1: 19.0, 2: 1.0}, ...], 'solver': ['lbfgs'], 'tol': [0.001, 0.0001]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation 

In [63]:
print(f"Further Best Logistic Regression Parameters found: {lr_rscv2.best_params_}")

Further Best Logistic Regression Parameters found: {'C': np.float64(0.4844998146905259), 'class_weight': {0: 13.0, 1: 19.0, 2: 1.0}, 'solver': 'lbfgs', 'tol': 0.0001}


In [64]:
lr_preds2 = lr_rscv.predict(X_test_tfidf)

In [65]:
print("---Logistic Regression 2---")
print(classification_report(y_test, lr_preds2))

---Logistic Regression 2---
              precision    recall  f1-score   support

           0       0.36      0.37      0.37       158
           1       0.14      0.32      0.20       157
           2       0.99      0.97      0.98     11302

    accuracy                           0.95     11617
   macro avg       0.50      0.55      0.52     11617
weighted avg       0.97      0.95      0.96     11617



Well.. nothing changed much.

In [66]:
# joblib.dump(lr_model, "models/sentiment_lr_model.pkl")
# joblib.dump(linearsvc_model, "models/sentiment_linearsvc_model.pkl")
# joblib.dump(vectorizer, "models/sentiment_vectorizer.pkl") 

Models trained in this file will not be pickled due to different PRNG from random_state